# Chapter 1: Introduction to Production Optimization with NeqSim

This notebook introduces the core concepts of production optimization for oil and gas fields,
and demonstrates how **NeqSim** — a Java-based thermodynamic and process simulation toolkit —
can be used to model fluid behavior, evaluate operating conditions, and support optimization workflows.

**Topics covered:**
- Creating a natural gas fluid model
- Running flash calculations at varying conditions
- Visualizing gas density, compressibility factor (Z), and phase envelopes
- Understanding how key parameters affect production rates

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


In [2]:
import matplotlib.pyplot as plt
import numpy as np

from neqsim import jneqsim

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

## 1.1 Creating a Natural Gas Fluid

We define a typical lean natural gas composition and create a thermodynamic system
using the Soave-Redlich-Kwong (SRK) equation of state.

In [3]:
def create_natural_gas(T_K, P_bara):
    """Create a natural gas fluid at given T (K) and P (bara)."""
    fluid = SystemSrkEos(T_K, P_bara)
    fluid.addComponent("methane", 0.85)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.04)
    fluid.addComponent("n-butane", 0.02)
    fluid.addComponent("CO2", 0.015)
    fluid.addComponent("nitrogen", 0.005)
    fluid.setMixingRule("classic")
    return fluid

# Quick test
fluid_test = create_natural_gas(273.15 + 25.0, 50.0)
ops_test = ThermodynamicOperations(fluid_test)
ops_test.TPflash()
fluid_test.initProperties()
print(f"Gas density at 25°C, 50 bara: {fluid_test.getDensity('kg/m3'):.2f} kg/m3")
print(f"Number of phases: {fluid_test.getNumberOfPhases()}")

Gas density at 25°C, 50 bara: 44.96 kg/m3
Number of phases: 1


## 1.2 Figure 1 — Gas Density vs Pressure

Gas density increases with pressure due to compression. At higher pressures the gas
deviates significantly from ideal gas behavior.

In [4]:
pressures = np.linspace(10, 200, 30)
densities = []
T_C = 25.0
T_K = 273.15 + T_C

for P in pressures:
    fluid = create_natural_gas(T_K, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()
    densities.append(fluid.getDensity("kg/m3"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pressures, densities, 'b-o', markersize=4, linewidth=2)
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Gas Density (kg/m³)', fontsize=12)
ax.set_title('Figure 1.1: Natural Gas Density vs Pressure at 25°C', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 210)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../figures/fig01_density_vs_pressure.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Density range: {min(densities):.1f} – {max(densities):.1f} kg/m³")

Density range: 8.1 – 204.1 kg/m³


C:\Users\ESOL\AppData\Local\Temp\ipykernel_37040\721819044.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.3 Figure 2 — Compressibility Factor (Z) vs Pressure

The Z-factor quantifies the deviation from ideal gas behavior (Z=1 for ideal gas).
At moderate pressures Z drops below 1 (attractive forces dominate), then rises
at very high pressures (repulsive forces).

In [5]:
z_factors = []

for P in pressures:
    fluid = create_natural_gas(T_K, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()
    z_factors.append(fluid.getPhase("gas").getZ())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pressures, z_factors, 'r-s', markersize=4, linewidth=2)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Ideal gas (Z=1)')
ax.set_xlabel('Pressure (bara)', fontsize=12)
ax.set_ylabel('Compressibility Factor Z (-)', fontsize=12)
ax.set_title('Figure 1.2: Z-Factor vs Pressure at 25°C', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 210)
plt.tight_layout()
plt.savefig('../figures/fig02_z_factor_vs_pressure.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Z-factor range: {min(z_factors):.4f} – {max(z_factors):.4f}")

Z-factor range: 0.7503 – 0.9744


C:\Users\ESOL\AppData\Local\Temp\ipykernel_37040\4032045061.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.4 Figure 3 — Phase Envelope

The phase envelope (P-T diagram) shows the bubble-point and dew-point curves
that bound the two-phase region. This is critical for understanding at which
conditions liquid may drop out from the gas.

In [6]:
fluid_env = create_natural_gas(273.15 + 25.0, 50.0)
fluid_env.setMultiPhaseCheck(True)
ops_env = ThermodynamicOperations(fluid_env)
ops_env.calcPTphaseEnvelope(True, 1.0)

# Extract dew and bubble point arrays
dewT_raw = ops_env.get("dewT")
dewP_raw = ops_env.get("dewP")
bubT_raw = ops_env.get("bubT")
bubP_raw = ops_env.get("bubP")

dewT = [dewT_raw[i] - 273.15 for i in range(dewT_raw.length)]
dewP = [dewP_raw[i] for i in range(dewP_raw.length)]
bubT = [bubT_raw[i] - 273.15 for i in range(bubT_raw.length)]
bubP = [bubP_raw[i] for i in range(bubP_raw.length)]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(dewT, dewP, 'b-', linewidth=2, label='Dew point curve')
ax.plot(bubT, bubP, 'r-', linewidth=2, label='Bubble point curve')
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Pressure (bara)', fontsize=12)
ax.set_title('Figure 1.3: Phase Envelope of Natural Gas Mixture', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('../figures/fig03_phase_envelope.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_37040\2971291532.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 1.5 Figure 4 — Production Rate Sensitivity

This conceptual figure illustrates how different operating parameters
(wellhead pressure, separator pressure, pipeline diameter, choke setting)
affect the achievable gas production rate. The relative impacts are estimated
from a simple choke/pipeline flow model.

In [7]:
# Sensitivity: compute gas density at different wellhead pressures as a proxy
# for production capacity (lower density = more volumetric throughput)
whp_pressures = [30, 50, 70, 90, 120]
flow_indices = []

for P in whp_pressures:
    fluid = create_natural_gas(273.15 + 40.0, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()
    rho = fluid.getDensity("kg/m3")
    # Simplified flow index: proportional to P / sqrt(rho * Z)
    z = fluid.getPhase("gas").getZ()
    flow_idx = P / (rho * z) ** 0.5
    flow_indices.append(flow_idx)

# Normalize to percentage of max
max_flow = max(flow_indices)
flow_pct = [100.0 * f / max_flow for f in flow_indices]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']
bars = ax.bar([str(p) for p in whp_pressures], flow_pct, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Wellhead Pressure (bara)', fontsize=12)
ax.set_ylabel('Relative Flow Index (%)', fontsize=12)
ax.set_title('Figure 1.4: Production Sensitivity to Wellhead Pressure', fontsize=13)
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')
for bar, pct in zip(bars, flow_pct):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1.5,
            f'{pct:.0f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../figures/fig04_production_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_37040\2314054609.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

This chapter demonstrated:

1. **Fluid creation** — Building a natural gas model with SRK EOS in NeqSim
2. **Density behavior** — Gas density increases non-linearly with pressure due to real-gas effects
3. **Z-factor** — The compressibility factor drops below 1.0 at moderate pressures, showing deviation from ideal gas
4. **Phase envelope** — The PT diagram shows where two-phase conditions may occur
5. **Production sensitivity** — Wellhead pressure significantly affects deliverability

In the next chapter we will explore the thermodynamic foundations (EOS models and flash calculations) in more detail.